# regsens Tutorial

**regsens** runs regression sensitivity analyses by fitting models for all
subsets of a set of variable predictors — cheaply.

The key ideas:

* **QR updating via Givens rotations.** When a predictor is added or removed,
  the thin QR factorization is updated in O(mp) instead of refactored from
  scratch in O(mp²). Subsets are visited in Gray code order so that
  consecutive models differ by exactly one predictor.
* **QR reuse across outcomes.** For linear models with multiple outcome
  columns, Q and R are factored once per subset and all outcomes are solved
  simultaneously.
* **IRLS warm-starting for GLMs.** The converged linear predictor from each
  model seeds the next model's IRLS, cutting iterations from ~10 to ~2.
* **Hierarchical parallelism.** 2^k subsets are split into chunks dispatched
  via `ProcessPoolExecutor`; each chunk runs its own sequential Gray code loop.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import regsens
from regsens import LinearSensitivity, GLMSensitivity

print(f"regsens {regsens.__version__}")

regsens 0.1.0


## 1. Synthetic data

We generate n=200 observations with **6 candidate predictors** (64 subsets).
Only x1 (β=1.5) and x3 (β=−1.0) are truly associated with the outcome.

In [ ]:
rng = np.random.default_rng(42)

n = 200
k = 6

# Moderately correlated design matrix
Sigma = 0.3 * np.ones((k, k)) + 0.7 * np.eye(k)
L = np.linalg.cholesky(Sigma)
X = rng.standard_normal((n, k)) @ L.T
variable_names = [f"x{i+1}" for i in range(k)]

# True model: x1 and x3 only
true_beta = np.array([1.5, 0.0, -1.0, 0.0, 0.0, 0.0])
y = X @ true_beta + rng.standard_normal(n)

print(f"X: {X.shape},  y: {y.shape}")
print(f"Subsets to evaluate: 2^{k} = {2**k}")

## 2. Linear sensitivity analysis

### 2.1 Fit all subsets

In [ ]:
sens = LinearSensitivity(
    X, y,
    variable_names=variable_names,
    intercept_always=True,  # intercept is always in — never toggled
    n_jobs=4,
)

result = sens.fit_all_subsets(return_se=True, return_pvalues=True)

print(f"Subsets evaluated : {result.n_subsets}")
print(f"Observations      : {result.n_obs}")
print(f"Elapsed           : {result.elapsed_sec:.2f}s")

### 2.2 Results as a DataFrame

**Wide format** — one row per subset, NaN for excluded predictors.

In [ ]:
df_wide = result.to_dataframe(format="wide")
print(f"Shape: {df_wide.shape}")
df_wide[["subset_id", "n_vars", "rss", "r2", "aic",
         "coef_(Intercept)", "coef_x1", "coef_x3"]].head(8)

**Long format** — one row per (subset, predictor), with an `included` column.

In [ ]:
df_long = result.to_dataframe(format="long")
print(f"Shape: {df_long.shape}")
print(f"Columns: {df_long.columns.tolist()}")
df_long.head(10)

### 2.3 Best subsets by AIC

In [ ]:
result.best_subsets(criterion="aic", n=8)

### 2.4 Coefficient stability

`variable_summary()` aggregates each predictor's coefficient across all
models in which it was included. Large SD signals sensitivity to model
specification.

In [ ]:
vs = result.variable_summary()
vs

In [ ]:
# Coefficient distribution across all subsets that include each variable.
# Long-format rows where `included=True` and predictor is one of our variables.
df_inc = df_long[df_long["included"] & df_long["predictor"].isin(variable_names)]

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, vname in zip(axes.flatten(), variable_names):
    coefs = df_inc.loc[df_inc["predictor"] == vname, "coef"]
    ax.hist(coefs, bins=20, edgecolor="white", linewidth=0.5)
    ax.axvline(0, color="red", linestyle="--", linewidth=1)
    ax.set_title(vname)
    ax.set_xlabel("coefficient")

plt.suptitle(
    "Coefficient distribution across all models that include each variable",
    y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# Stability summary: mean ± SD across models
vs_var = vs[vs["variable"].isin(variable_names)].set_index("variable")

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(
    x=range(len(vs_var)),
    y=vs_var["coef_mean"],
    yerr=vs_var["coef_sd"],
    fmt="o", capsize=5, linewidth=2, markersize=7
)
ax.axhline(0, color="gray", linestyle="--")
ax.set_xticks(range(len(vs_var)))
ax.set_xticklabels(vs_var.index)
ax.set_ylabel("Coefficient (mean ± SD across models)")
ax.set_title("Coefficient stability across all-subsets")
plt.tight_layout()
plt.show()

### 2.5 R² by model size

In [ ]:
r2_by_size = (
    df_wide[["n_vars", "r2"]]
    .dropna()
    .groupby("n_vars")["r2"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(r2_by_size["n_vars"], r2_by_size["min"], r2_by_size["max"],
                alpha=0.25, label="min–max range")
ax.plot(r2_by_size["n_vars"], r2_by_size["mean"], "o-", label="mean R²")
ax.set_xlabel("Number of variable predictors")
ax.set_ylabel("R²")
ax.set_title("R² by model size")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Multiple outcomes

For a fixed predictor set the QR factorization is computed once and all
outcome columns are solved in one triangular solve — no extra factorizations.

In [ ]:
B_true = np.array([
    [1.5,  0.0, -1.0, 0.0, 0.0, 0.0],   # y1: x1, x3
    [0.0,  2.0,  0.0, 0.0, 0.0, 0.0],   # y2: x2 only
    [0.5,  0.0,  0.5, 0.0, 0.8, 0.0],   # y3: x1, x3, x5
]).T  # (k, 3)

Y = X @ B_true + rng.standard_normal((n, 3))
outcome_names = ["y1", "y2", "y3"]

sens_multi = LinearSensitivity(
    X, Y=Y,
    variable_names=variable_names,
    outcome_names=outcome_names,
)

result_multi = sens_multi.fit_all_subsets(return_se=True)
df_multi = result_multi.to_dataframe(format="wide")

print(f"Shape: {df_multi.shape}")
# x1 coefficient column for each outcome
x1_cols = [c for c in df_multi.columns if c.startswith("coef_x1")]
print("x1 coef columns:", x1_cols)
df_multi[["n_vars"] + x1_cols].head(8)

## 4. GLM sensitivity analysis

### 4.1 Logistic regression (binary outcome)

In [ ]:
eta_true = 0.8 * X[:, 0] - 0.6 * X[:, 2]
prob = 1 / (1 + np.exp(-eta_true))
y_bin = rng.binomial(1, prob).astype(float)

print(f"Event rate: {y_bin.mean():.2%}")

In [ ]:
sens_glm = GLMSensitivity(
    X, y_bin,
    family="binomial",
    variable_names=variable_names,
    warm_start=True,  # pass converged eta to neighbouring models
    n_jobs=4,
)

result_glm = sens_glm.fit_all_subsets()
print(f"Subsets evaluated : {result_glm.n_subsets}")
print(f"Elapsed           : {result_glm.elapsed_sec:.2f}s")

In [ ]:
df_glm = result_glm.to_dataframe(format="wide")
df_glm[["n_vars", "deviance", "aic", "pseudo_r2",
        "coef_(Intercept)", "coef_x1", "coef_x3"]].head(10)

### 4.2 Best subsets by AIC (logistic)

In [ ]:
result_glm.best_subsets(criterion="aic", n=8)

### 4.3 McFadden pseudo-R² by model size

In [ ]:
r2g = (
    df_glm[["n_vars", "pseudo_r2"]]
    .dropna()
    .groupby("n_vars")["pseudo_r2"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(r2g["n_vars"], r2g["min"], r2g["max"],
                alpha=0.25, color="tomato", label="min–max range")
ax.plot(r2g["n_vars"], r2g["mean"], "o-", color="tomato",
        label="mean McFadden R²")
ax.set_xlabel("Number of variable predictors")
ax.set_ylabel("McFadden pseudo-R²")
ax.set_title("McFadden R² by model size (logistic)")
ax.legend()
plt.tight_layout()
plt.show()

### 4.4 Poisson regression (count outcome)

In [ ]:
mu_pois = np.exp(0.4 + 0.6 * X[:, 0] - 0.4 * X[:, 2])
y_count = rng.poisson(mu_pois).astype(float)

sens_pois = GLMSensitivity(
    X, y_count,
    family="poisson",
    variable_names=variable_names,
    warm_start=True,
)

result_pois = sens_pois.fit_all_subsets()
result_pois.best_subsets(criterion="aic", n=5)

## 5. Leave-one-out and add-one

These conveniences fit exactly k models — useful for a quick stability
check without running the full 2^k enumeration.

In [ ]:
# Leave-one-out from the full model
loo = sens.fit_leave_one_out()
df_loo = loo.to_dataframe(format="wide")
df_loo[["n_vars", "r2", "aic"]].assign(dropped=variable_names)

In [ ]:
# Add-one from the intercept-only model
ao = sens.fit_add_one()
df_ao = ao.to_dataframe(format="wide")
df_ao[["n_vars", "r2", "aic"]].assign(added=variable_names)

## 6. Single-subset inspection

`fit_subset` fits exactly one model — no Gray code traversal, no
parallelism overhead.

In [ ]:
mask = [True, False, True, False, False, False]   # x1 + x3 only
r = sens.fit_subset(mask)

active = [nm for nm, inc in zip(variable_names, mask) if inc]
all_terms = ["(Intercept)"] + active

pd.DataFrame({
    "term": all_terms,
    "coef": r["coef"].round(4),
    "se":   r["se"].round(4),
    "pval": r["pvalues"].round(4),
})

## 7. Under the hood: Gray code traversal

Consecutive Gray code values differ by exactly one bit, so each model
transition requires only a single column add or remove.

In [ ]:
from regsens.gray_code import gray, gray_diff, gray_to_mask

k_demo = 3
print(f"{'i':>3}  {'G(i)':>5}  {'mask':<8}  change")
print("-" * 38)
for i in range(2**k_demo):
    g = gray(i)
    mask_str = "".join(str(int(b)) for b in gray_to_mask(g, k_demo))
    if i == 0:
        change = "(start)"
    else:
        bp, is_add = gray_diff(i)
        change = f"{'add' if is_add else 'remove'} x{bp+1}"
    print(f"{i:>3}  {g:>5}  {mask_str:<8}  {change}")

## 8. Performance: QR update vs fresh factorization

The QR column-add (Gram-Schmidt extension) is O(mp); a fresh factorization
of the same matrix is O(mp²). The speedup grows with p.

In [2]:
import time
from regsens.qr_core import qr_state_from_cols, qr_add_column

m_bench = 5_000
p_bench = 20
rng_b   = np.random.default_rng(0)
X_big   = rng_b.standard_normal((m_bench, p_bench + 1))

state = qr_state_from_cols(X_big, list(range(p_bench)))
z_new = X_big[:, p_bench]
n_reps = 500

t0 = time.perf_counter()
for _ in range(n_reps):
    _ = qr_add_column(state, z_new, p_bench, inplace=False)
t_add = (time.perf_counter() - t0) / n_reps * 1000

t0 = time.perf_counter()
for _ in range(n_reps):
    _ = qr_state_from_cols(X_big, list(range(p_bench + 1)))
t_fresh = (time.perf_counter() - t0) / n_reps * 1000

print(f"QR add (update)   : {t_add:.3f} ms")
print(f"QR fresh          : {t_fresh:.3f} ms")
print(f"Speedup           : {t_fresh / t_add:.1f}×  (m={m_bench}, p={p_bench})")

QR add (update)   : 0.137 ms
QR fresh          : 1.481 ms
Speedup           : 10.8×  (m=5000, p=20)
